# 🎬 Multimodal Viral Clip Discovery & Editorial Intelligence Workbench
### *(AI Creative Director Co-Pilot)*

**Author:** Sonali Gupta (CU23BCA0060A) | BCA, Chanakya University | **Guide:** Miss Vathsala Bhat

**Output:** an `editorial_suggestions.csv` timeline (start/end timestamps, jump links, category, score with explanation, music mood, mixing cues, framing notes, keywords, transcript, per-second music-intensity recommendation). No clip files are produced.

**Version 3:** fixes the Hugging Face 401 error (stale token ignored), adds a model fallback, and launches through a Cloudflare tunnel.

**Two-pass design:** (1) audio-only download, transcription, semantic + acoustic ranking; (2) only the shortlisted intervals are fetched at <=360p for frame inspection, then deleted. All temporary files are removed in a `finally` block.

> Set **Runtime > Change runtime type > T4 GPU** before running. Run the cells top to bottom, then open the link printed by the last cell.

## Step 1: Install dependencies

In [ ]:
import os
for _k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
    os.environ.pop(_k, None)          # ignore any stale Hugging Face token
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
!rm -f ~/.cache/huggingface/token ~/.huggingface/token
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q -U yt-dlp
!pip install -q streamlit faster-whisper transformers torch keybert librosa opencv-python-headless plotly pandas scipy sentencepiece
!mkdir -p src && touch src/__init__.py
print('Dependencies ready')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Dependencies ready


## Step 2: Backend modules (`src/`)

In [ ]:
%%writefile src/ingestion.py
import glob
import os
import subprocess

import yt_dlp


def cookie_opt():
    """Use cookies.txt (if present) so yt-dlp can pass YouTube's bot checks."""
    return {"cookiefile": "cookies.txt"} if os.path.exists("cookies.txt") else {}


def fetch_youtube_metadata_and_audio(url: str, session_dir: str):
    """Pass 1: metadata + audio-only download (lightweight)."""
    base = {"quiet": True, "no_warnings": True, "noplaylist": True, **cookie_opt()}
    with yt_dlp.YoutubeDL(base) as ydl:
        info = ydl.extract_info(url, download=False)
    vid_id = info.get("id", "")

    opts = {**base, "format": "bestaudio/best",
            "outtmpl": os.path.join(session_dir, "raw_audio.%(ext)s")}
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])
    raw = glob.glob(os.path.join(session_dir, "raw_audio.*"))[0]

    return {
        "title": info.get("title", "YouTube Video"),
        "channel": info.get("uploader", "Unknown"),
        "duration": info.get("duration", 0),
        "base_link": f"https://youtu.be/{vid_id}" if vid_id else url,
        "raw_media_path": raw,
    }


def download_section_lowres(url: str, start: float, end: float, out_dir: str, tag: str):
    """Pass 2: download ONLY one selected interval at <=360p, used for frame inspection."""
    opts = {
        "format": "best[height<=360]/worst",
        "outtmpl": os.path.join(out_dir, f"{tag}.%(ext)s"),
        "download_ranges": yt_dlp.utils.download_range_func(None, [(start, end)]),
        "quiet": True, "no_warnings": True, "noplaylist": True, **cookie_opt(),
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])
    files = glob.glob(os.path.join(out_dir, f"{tag}.*"))
    return files[0] if files else None


def convert_to_16k_mono_wav(input_path: str, wav_path: str):
    """FFmpeg: any audio/video -> 16 kHz mono WAV for faster-whisper."""
    subprocess.run(["ffmpeg", "-y", "-i", input_path, "-vn", "-ac", "1", "-ar", "16000", wav_path],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return wav_path


Overwriting src/ingestion.py


In [ ]:
%%writefile src/transcription.py
def load_whisper_model(device: str, compute_type: str, size: str = "base"):
    from faster_whisper import WhisperModel
    return WhisperModel(size, device=device, compute_type=compute_type)


def _make_window(cur):
    return {"start": cur[0].start, "end": cur[-1].end,
            "text": " ".join(s.text.strip() for s in cur)}


def group_segments(segments, min_dur: float = 30.0, max_dur: float = 60.0):
    """Group transcript segments into 30-60 s windows, preferring to end on a sentence boundary."""
    windows, cur = [], []
    for seg in segments:
        if cur and seg.end - cur[0].start > max_dur:
            # hard cap reached without a sentence end: close what we have if long enough
            if cur[-1].end - cur[0].start >= min_dur:
                windows.append(_make_window(cur))
            cur = []
        cur.append(seg)
        if seg.end - cur[0].start >= min_dur and seg.text.strip().endswith((".", "?", "!")):
            windows.append(_make_window(cur))
            cur = []
    if cur and cur[-1].end - cur[0].start >= min_dur:
        windows.append(_make_window(cur))
    return windows


def transcribe_and_segment(whisper_model, wav_path: str, min_dur: float = 30.0, max_dur: float = 60.0):
    segments, _ = whisper_model.transcribe(wav_path, beam_size=1, vad_filter=True)
    return group_segments(list(segments), min_dur, max_dur)


Overwriting src/transcription.py


In [ ]:
%%writefile src/analysis.py
import re

import cv2
import librosa
import numpy as np

SR = 16000
W_SEM, W_AC = 0.60, 0.25          # linguistic boost adds up to 0.15 -> pre-score max 1.0
W_PRE, W_VIS = 0.90, 0.10         # final = 0.9*pre-score + 0.1*face presence (shortlist only)

CANDIDATE_LABELS = ["counter-intuitive insight", "bold opinion", "actionable advice",
                    "personal story", "general filler"]
MOOD = {
    "counter-intuitive insight": "Curious, suspenseful build",
    "bold opinion": "Tense, driving beat",
    "actionable advice": "Upbeat, motivational",
    "personal story": "Warm, emotional piano/strings",
    "general filler": "Neutral / minimal music",
}


def load_analysis_models(device: str):
    from transformers import pipeline
    from keybert import KeyBERT
    primary = "MoritzLaurer/DeBERTa-v3-small-mnli-fever-anli-ling-wanli"
    fallback = "facebook/bart-large-mnli"      # only used if the DeBERTa model cannot be loaded
    dev = 0 if device == "cuda" else -1
    try:
        clf_name = primary
        clf = pipeline("zero-shot-classification", model=primary, device=dev)
    except Exception as err:
        print("Primary model failed, using fallback:", repr(err)[:200])
        clf_name = fallback
        clf = pipeline("zero-shot-classification", model=fallback, device=dev)
    kw = KeyBERT()
    face = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    return clf, kw, face, clf_name


def linguistic_boost(text: str) -> float:
    """Supporting cue only: curiosity words and a question mark (max +0.15)."""
    words = r"\b(why|how|secret|mistake|truth|never|nobody|actually|surprising|reason)\b"
    hits = len(re.findall(words, text.lower()))
    return min(hits * 0.03 + (0.05 if "?" in text else 0.0), 0.15)


def analyze_acoustics(y, start_s: float, end_s: float):
    """Energy variation, pauses and a per-second music-intensity RECOMMENDATION (1-10).
    Music is suggested low under loud speech and higher in quiet moments (ducking logic)."""
    seg = y[int(start_s * SR):int(end_s * SR)]
    n = max(int(end_s - start_s), 1)
    if len(seg) < SR:
        return {"acoustic": 0.0, "pauses": 0, "curve": [5] * n, "quiet_sec": 0, "peak_sec": 0}

    rms = librosa.feature.rms(y=seg, frame_length=SR, hop_length=SR)[0][:n]
    norm = (rms - rms.min()) / (np.ptp(rms) + 1e-9)
    level = np.clip(1.0 - 0.6 * norm, 0.2, 0.95)
    curve = np.round(level * 10).astype(int).tolist()

    intervals = librosa.effects.split(seg, top_db=30)
    gaps = [(intervals[i + 1][0] - intervals[i][1]) / SR for i in range(len(intervals) - 1)]
    pauses = sum(1 for g in gaps if g >= 0.4)

    dyn = float(np.std(norm))
    acoustic = 0.5 * min(dyn * 2.5, 1.0) + 0.5 * min(pauses, 5) / 5
    return {"acoustic": round(acoustic, 3), "pauses": pauses, "curve": curve,
            "quiet_sec": int(np.argmin(norm)), "peak_sec": int(np.argmax(norm))}


def analyze_visual_framing(video_path: str, face_detector, start_s: float, end_s: float, n_frames: int = 6):
    """Sample frames INSIDE [start_s, end_s] of the video and detect faces (approximate cue only)."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, "Video unavailable: use B-roll/captions; apply a standard 9:16 centre crop."

    counts, areas, diffs, prev = [], [], [], None
    for t in np.linspace(start_s, max(end_s - 0.5, start_s), n_frames):
        cap.set(cv2.CAP_PROP_POS_MSEC, float(t) * 1000)
        ok, frame = cap.read()
        if not ok:
            continue
        h, w = frame.shape[:2]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if prev is not None:
            diffs.append(float(np.mean(cv2.absdiff(gray, prev))))
        prev = gray
        faces = face_detector.detectMultiScale(gray, 1.1, 4)
        counts.append(len(faces))
        if len(faces):
            areas.append(max(fw * fh for (_, _, fw, fh) in faces) / float(w * h))
    cap.release()

    if not counts:
        return None, "Frames could not be read: apply a standard 9:16 centre crop."

    presence = sum(1 for c in counts if c > 0) / len(counts)
    avg_faces = float(np.mean(counts))
    avg_area = float(np.mean(areas)) if areas else 0.0
    motion = float(np.mean(diffs)) if diffs else 0.0

    if avg_faces >= 1.5:
        shot = "Two-shot/group frame: consider a split-screen or wider 9:16 crop."
    elif 0 < avg_area < 0.08:
        shot = "Solo medium/wide shot: a modest 105-110% punch-in on key emphasis may work."
    elif avg_area >= 0.08:
        shot = "Tight close-up: keep framing steady; add animated subtitles."
    else:
        shot = "No clear face detected: consider B-roll or text overlays."
    move = ("Static camera: add B-roll/text every 8-10 s." if motion < 8.0
            else "Visual cuts present: keep edits minimal.")
    return round(presence, 2), f"{shot} {move} (approximate; face detection does not identify the speaker)"


def rank_candidate_windows(windows, y, zero_shot_clf):
    """Pass 1 scoring from text + audio only (no video needed)."""
    scored = []
    for w in windows:
        res = zero_shot_clf(w["text"][:1000], candidate_labels=CANDIDATE_LABELS)
        probs = dict(zip(res["labels"], res["scores"]))
        sem = 1.0 - float(probs["general filler"])
        ling = linguistic_boost(w["text"])
        ac = analyze_acoustics(y, w["start"], w["end"])
        pre = min(W_SEM * sem + W_AC * ac["acoustic"] + ling, 1.0)
        w.update({"category": res["labels"][0], "sem": round(sem, 3), "ling": round(ling, 3),
                  "ac": ac, "pre_score": round(pre, 3)})
        scored.append(w)
    return sorted(scored, key=lambda x: x["pre_score"], reverse=True)


Overwriting src/analysis.py


In [ ]:
%%writefile src/packaging.py
import os

import pandas as pd

from src.analysis import MOOD, W_AC, W_PRE, W_SEM, W_VIS, analyze_visual_framing
from src.ingestion import download_section_lowres


def fmt_time(sec: float) -> str:
    sec = int(sec)
    return f"{sec // 3600:02d}:{sec % 3600 // 60:02d}:{sec % 60:02d}"


def build_editorial_table(top, session_dir, kw_extractor, face_detector,
                          youtube_url=None, local_video_path=None, base_link=""):
    """Pass 2: for the shortlisted windows only, inspect frames (low-res section for YouTube),
    then build the editorial timeline table. No clip files are produced."""
    rows = []
    for i, c in enumerate(top):
        face, vnote = None, "No video stream available: use captions/B-roll."
        try:
            if local_video_path:
                face, vnote = analyze_visual_framing(local_video_path, face_detector, c["start"], c["end"])
            elif youtube_url:
                sec_file = download_section_lowres(youtube_url, c["start"], c["end"], session_dir, f"sec_{i}")
                if sec_file:
                    face, vnote = analyze_visual_framing(sec_file, face_detector, 0, c["end"] - c["start"])
                    os.remove(sec_file)
        except Exception:
            vnote = "Video section could not be retrieved (restricted): use the jump link and cut manually."

        final = c["pre_score"] if face is None else W_PRE * c["pre_score"] + W_VIS * face
        ac = c["ac"]
        kws = [k for k, _ in kw_extractor.extract_keywords(
            c["text"], keyphrase_ngram_range=(1, 2), stop_words="english", top_n=4)]
        visual_part = "visual unavailable" if face is None else f"face presence {face:.2f} x {W_VIS}"
        rows.append({
            "start_sec": round(c["start"], 1), "end_sec": round(c["end"], 1),
            "start_time": fmt_time(c["start"]), "end_time": fmt_time(c["end"]),
            "duration_sec": round(c["end"] - c["start"], 1),
            "jump_link": f"{base_link}?t={int(c['start'])}" if base_link else "Local file",
            "category": c["category"],
            "score": round(final, 3),
            "rationale": (f"semantic {c['sem']} x {W_SEM} + acoustic {ac['acoustic']} x {W_AC} + "
                          f"linguistic {c['ling']} = {c['pre_score']}; final = {W_PRE} x that + {visual_part}"),
            "recommended_music_mood": MOOD[c["category"]],
            "editing_cues": (f"0s: bold hook caption + whoosh SFX | {ac['quiet_sec']}s: transition / music swell "
                             f"(quietest moment) | {ac['peak_sec']}s: punch-in on emphasis"),
            "audio_mixing_cue": ("Several pauses: use them for transitions or SFX hits"
                                 if ac["pauses"] >= 3 else "Few pauses: keep music low (about -18 to -20 dB) under dialogue"),
            "visual_framing_notes": vnote,
            "keywords": ", ".join(kws),
            "transcript_excerpt": c["text"],
            "music_intensity_curve": " ".join(map(str, ac["curve"])),
            "curve": ac["curve"],
        })

    df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    df.insert(0, "rank", df.index + 1)
    csv_bytes = df.drop(columns=["curve"]).to_csv(index=False).encode("utf-8")
    return df, csv_bytes


Overwriting src/packaging.py


## Step 3: Streamlit interface (`app.py`)

In [ ]:
%%writefile app.py
import os
for _k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
    os.environ.pop(_k, None)          # ignore any stale Hugging Face token
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
import shutil
import tempfile

import librosa
import plotly.express as px
import streamlit as st
import torch

from src.analysis import load_analysis_models, rank_candidate_windows
from src.ingestion import convert_to_16k_mono_wav, fetch_youtube_metadata_and_audio
from src.packaging import build_editorial_table
from src.transcription import load_whisper_model, transcribe_and_segment

st.set_page_config(page_title="AI Creative Director Co-Pilot", layout="wide")
st.title("🎬 Multimodal Viral Clip Discovery & Editorial Intelligence Workbench")
st.caption("Decision-support tool: ranks promising segments and suggests edits. It does not predict virality.")

with st.sidebar:
    st.header("⚙️ Ingestion Settings")
    st.caption("If YouTube blocks the download, upload cookies.txt from a throwaway Google account, "
               "or use the local file option. cookies.txt is deleted after each run.")
    cookie = st.file_uploader("Optional: cookies.txt", type=["txt"])
    if cookie is not None:
        with open("cookies.txt", "wb") as f:
            f.write(cookie.getbuffer())
        st.success("cookies.txt loaded for the next run")


@st.cache_resource
def init_models():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute = "float16" if device == "cuda" else "int8"
    whisper = load_whisper_model(device, compute, "small" if device == "cuda" else "base")
    clf, kw, face, clf_name = load_analysis_models(device)
    return whisper, clf, kw, face, clf_name


with st.spinner("Loading local models (Whisper, DeBERTa-v3, KeyBERT, OpenCV)..."):
    whisper_model, clf, kw_extractor, face_detector, clf_name = init_models()
st.sidebar.caption(f'Semantic model: {clf_name}')

mode = st.radio("Video source:", ["Public YouTube URL", "Upload video/audio file"], horizontal=True)
youtube_url, uploaded = None, None
if mode == "Public YouTube URL":
    youtube_url = st.text_input("Public YouTube URL:", placeholder="https://www.youtube.com/watch?v=...")
else:
    uploaded = st.file_uploader("Upload podcast video/audio:", type=["mp4", "mkv", "mov", "mp3", "wav", "m4a"])
num_clips = st.slider("Number of candidate segments:", 1, 10, 5)


def show_results(res):
    df = res["df"]
    st.success(f"Done: {res['title']}")
    st.download_button("⬇️ Download editorial_suggestions.csv", res["csv"],
                       file_name="editorial_suggestions.csv", mime="text/csv")
    st.subheader("Editorial timeline")
    st.dataframe(df.drop(columns=["curve", "music_intensity_curve", "transcript_excerpt"]))
    st.subheader("Per-segment details")
    for idx, row in df.iterrows():
        title = f"#{row['rank']}  {row['start_time']} - {row['end_time']}  |  {row['category'].title()}  |  score {row['score']}"
        with st.expander(title, expanded=(idx == 0)):
            a, b = st.columns(2)
            with a:
                st.markdown(f"**Jump link:** {row['jump_link']}")
                st.markdown(f"**Why ranked here:** {row['rationale']}")
                st.markdown(f"**Editing cues:** {row['editing_cues']}")
                st.markdown(f"**Audio mixing:** {row['audio_mixing_cue']}")
                st.markdown(f"**Music mood (suggestion only):** {row['recommended_music_mood']}")
                st.markdown(f"**Visual framing:** {row['visual_framing_notes']}")
                st.markdown(f"**Keywords:** {row['keywords']}")
            with b:
                fig = px.line(x=list(range(len(row["curve"]))), y=row["curve"],
                              labels={"x": "Second in segment", "y": "Suggested music intensity (1-10)"},
                              title="Music-intensity recommendation (rule-based, not generated audio)")
                fig.update_yaxes(range=[0, 10.5])
                st.plotly_chart(fig)
                st.info(row["transcript_excerpt"])


if st.button("Generate editorial blueprint", type="primary") and (youtube_url or uploaded):
    session_dir = tempfile.mkdtemp(prefix="workbench_")
    try:
        with st.status("Running multimodal pipeline...", expanded=True) as status:
            wav_path = os.path.join(session_dir, "audio.wav")
            local_video, title, base_link = None, "Uploaded media", ""

            if youtube_url:
                st.write("1. Fetching metadata and audio...")
                try:
                    meta = fetch_youtube_metadata_and_audio(youtube_url, session_dir)
                    title, base_link = meta["title"], meta["base_link"]
                    st.caption(f"{title} | {meta['channel']} | {meta['duration']} s")
                    convert_to_16k_mono_wav(meta["raw_media_path"], wav_path)
                except Exception as err:
                    st.error(f"YouTube download failed ({err}). Try cookies.txt or the upload option.")
                    st.stop()
            else:
                st.write("1. Processing uploaded file...")
                title = uploaded.name
                in_path = os.path.join(session_dir, uploaded.name)
                with open(in_path, "wb") as f:
                    f.write(uploaded.getbuffer())
                if uploaded.name.lower().endswith((".mp4", ".mkv", ".mov")):
                    local_video = in_path
                convert_to_16k_mono_wav(in_path, wav_path)

            st.write("2. Transcribing locally (faster-whisper)...")
            windows = transcribe_and_segment(whisper_model, wav_path)
            if not windows:
                st.error("Not enough continuous speech to form 30-60 s windows.")
                st.stop()

            st.write(f"3. Scoring {len(windows)} windows (DeBERTa-v3 + linguistic + acoustic)...")
            y, _ = librosa.load(wav_path, sr=16000)
            ranked = rank_candidate_windows(windows, y, clf)
            shortlist = ranked[:num_clips]

            st.write("4. Inspecting frames of the shortlisted segments and building the blueprint...")
            df, csv_bytes = build_editorial_table(shortlist, session_dir, kw_extractor, face_detector,
                                                  youtube_url, local_video, base_link)
            st.session_state["result"] = {"df": df, "csv": csv_bytes, "title": title}
            status.update(label=f"Complete: {title}", state="complete", expanded=False)
    except Exception as e:
        st.error(f"Pipeline error: {e}")
    finally:
        shutil.rmtree(session_dir, ignore_errors=True)
        if os.path.exists("cookies.txt"):
            os.remove("cookies.txt")

if "result" in st.session_state:
    show_results(st.session_state["result"])


Overwriting app.py


In [ ]:
p = "src/transcription.py"
s = open(p).read()
old = 'segments, _ = whisper_model.transcribe(wav_path, beam_size=1, vad_filter=True)'
new = ('import librosa\n'
       '    audio, _ = librosa.load(wav_path, sr=16000, mono=True)\n'
       '    segments, _ = whisper_model.transcribe(audio, beam_size=1, vad_filter=True)')
assert old in s, "line not found - check src/transcription.py"
open(p, "w").write(s.replace(old, new))
print("patched")

patched


In [ ]:
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import os, glob, site
libs = []
for sp in site.getsitepackages():
    libs += glob.glob(f"{sp}/nvidia/*/lib")
os.environ["LD_LIBRARY_PATH"] = ":".join(libs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("Added", len(libs), "library folders")

Added 14 library folders


## Step 4: Launch the app
If the first model load looks stuck, wait: Whisper and DeBERTa download on first use.

In [ ]:
import os, re, subprocess, time

# Clear stale Hugging Face tokens for the Streamlit process too
for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
    os.environ.pop(k, None)
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# 1. Syntax check
!python -m py_compile src/*.py app.py && echo "Syntax OK"

# 2. Restart Streamlit cleanly
!pkill -f "streamlit run" || true
!pkill -f cloudflared || true
time.sleep(2)
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0",
                  "--server.headless", "true", "--server.enableCORS", "false",
                  "--server.enableXsrfProtection", "false"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

# 3. Public link via Cloudflare tunnel (works in any browser, no password)
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        print("\nOPEN THIS LINK:", m.group(0))
        print("(First page load takes 1-2 minutes while models download. Keep this Colab tab open.)")
        break
else:
    print("Tunnel did not start. Log:\n", open("tunnel.log").read())
    print("\nStreamlit log:\n", open("streamlit.log").read()[-1500:])


Syntax OK
^C
^C

OPEN THIS LINK: https://soccer-rating-cio-livestock.trycloudflare.com
(First page load takes 1-2 minutes while models download. Keep this Colab tab open.)
